# Imports and Setup


In [0]:
from pyspark.sql.functions import *
from delta.tables import DeltaTable
from datetime import datetime
import uuid

In [0]:
spark.sql("""
          create catalog if not exists novocart_catalog
          managed location 'abfss://novocartdata@udepracticesa.dfs.core.windows.net/'
          """)

In [0]:
spark.sql("""
          use catalog novocart_catalog
          """)

In [0]:
spark.sql("""
          create schema if not exists bronze_schema
          """)

# Bronze Control Table

In [0]:
spark.sql("""
          create table if not exists novocart_catalog.bronze_schema.ingestion_control (
              layer string,
              table_name string,
              ts_col string,
              pk_col string,
              last_successfull_ts timestamp,
              last_successfull_pk bigint,
              last_run_id string,
              rows_written bigint,
              run_status string,
              updated_at timestamp
          )
          using delta
          """)

# Source table configuration

In [0]:
tables_config = {
    "orders" : {"pk_col":"order_id", "ts_col":"updated_at"},
    "products" : {"pk_col":"product_id", "ts_col":"updated_at"},
    "payments" : {"pk_col":"payment_id", "ts_col":"processed_at"},
}

bronze_run_id = str(uuid.uuid4())
print("Current Bronze Run ID: ", bronze_run_id)

# Helper Functions

In [0]:
def get_last_successful_watermark(table_name: str):
    ctrl = (
        spark.table("novocart_catalog.bronze_schema.ingestion_control")
        .filter(
            (col("layer") == "bronze")
            & (col("table_name") == table_name)
            & (col("run_status") == "success")
        )
        .orderBy(col("updated_at").desc())
        .limit(1)
    )

    rows = ctrl.collect()

    if not rows:
        return None, None
    else:
        return rows[0]["last_successfull_ts"], rows[0]["last_successfull_pk"]

In [0]:
def upsert_bronze_control(
    table_name: str, ts_col, pk_col, last_ts, last_pk, rows_written, run_id
):
    control_df = spark.createDataFrame(
        [
            (
                "bronze",
                table_name,
                ts_col,
                pk_col,
                last_ts,
                int(last_pk) if last_pk is not None else None,
                run_id,
                int(rows_written),
                "success",
                datetime.utcnow(),
            )
        ],
        schema="""
        layer string, table_name string,
        ts_col string, pk_col string,
        last_successfull_ts timestamp, last_successfull_pk bigint,
        last_run_id string, rows_written bigint,
        run_status string, updated_at timestamp
        """,
    )

    dt = DeltaTable.forName(spark, "novocart_catalog.bronze_schema.ingestion_control")

    (
        dt.alias("t")
        .merge(
            control_df.alias("s"), "t.table_name = s.table_name and t.layer = s.layer"
        )
        .whenMatchedUpdate(
            set={
                "ts_col": "s.ts_col",
                "pk_col": "s.pk_col",
                "last_successfull_ts": "s.last_successfull_ts",
                "last_successfull_pk": "s.last_successfull_pk",
                "last_run_id": "s.last_run_id",
                "rows_written": "s.rows_written",
                "run_status": "s.run_status",
                "updated_at": "s.updated_at",
            }
        )
        .whenNotMatchedInsertAll()
        .execute()
    )

# Bronze Incremental Load Loop

In [0]:
for table_name, cfg in tables_config.items():
    pk_col = cfg["pk_col"]
    ts_col = cfg["ts_col"]

    source_table = f"`datacrunch_novocart_sql_connection_catalog`.dbo.{table_name}"
    target_table = f"novocart_catalog.bronze_schema.{table_name}_raw"
    last_successful_ts, last_successful_pk = get_last_successful_watermark(table_name)
    print(f"We are processing {table_name}")
    print("Last Successful TS: ", last_successful_ts)
    print("Last Successful PK: ", last_successful_pk)

    source_df = spark.read.table(source_table).withColumn(
        ts_col, col(ts_col).cast("timestamp")
    )

    if last_successful_ts is None:
        rows_to_load = source_df
    else:
        rows_to_load = source_df.filter(
            (col(ts_col) > lit(last_successful_ts))
            | (
                (col(ts_col) == lit(last_successful_ts))
                & (col(pk_col) > lit(int(last_successful_pk)))
            )
        )

    rows_to_load = (
        rows_to_load.withColumn("bronze_ingested_at", lit(current_timestamp()))
        .withColumn("bronze_run_id", lit(bronze_run_id))
        .withColumn("bronze_source_table", lit(source_table))
    )

    rows_count = rows_to_load.count()
    print(f"{table_name} rows_to_load = {rows_count}")

    if rows_count == 0:
        print(f"No new rows to load for {table_name}.")
        upsert_bronze_control(
            table_name,
            ts_col,
            pk_col,
            last_successful_ts,
            last_successful_pk,
            rows_count,
            bronze_run_id,
        )
        continue

    rows_to_load.write.format("delta").mode("overwrite").saveAsTable(target_table)

    max_ts = rows_to_load.agg(max(col(ts_col)).alias("max_ts")).collect()[0][
        "max_ts"
    ]
    max_pk = (
        rows_to_load.filter(col(ts_col) == max_ts)
        .agg(max(col(pk_col)).cast("long").alias("max_pk"))
        .collect()[0]["max_pk"]
    )

    upsert_bronze_control(table_name, ts_col, pk_col, max_ts, max_pk, rows_count, bronze_run_id)
    print(f"wrote {rows_count} rows to {target_table}")

# Quick Validation

In [0]:
print("Orders Bronze Count: ", spark.sql("select count(*) from novocart_catalog.bronze_schema.orders_raw").collect()[0][0])
print("Products Bronze Count: ", spark.sql("select count(*) from novocart_catalog.bronze_schema.products_raw").collect()[0][0])
print("Payments Bronze Count: ", spark.sql("select count(*) from novocart_catalog.bronze_schema.payments_raw").collect()[0][0])

display(spark.sql("select * from novocart_catalog.bronze_schema.ingestion_control").orderBy(col("table_name")))